# Week 07 · Friday — ShopSense Sentiment Analysis
**PG Diploma · AI-ML & Agentic AI Engineering · IIT Gandhinagar**

> Topics: NLP Evaluation · ML Pitfalls · Model Selection · Production Readiness  
> Dataset: ShopSense Reviews (`product_reviews.csv`)

---


In [ ]:
# ─────────────────────────── Imports ───────────────────────────────────────────
import os, time, warnings, re, string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ─────────────────────────── CONFIG (no magic numbers) ─────────────────────────
DATA_PATH               = "product_reviews.csv"   # relative path — keep CSV here
RANDOM_STATE            = 42
TEST_SIZE               = 0.20    # 80/20 split

# Sentiment score buckets
NEGATIVE_SCORES         = [1, 2]
NEUTRAL_SCORES          = [3]
POSITIVE_SCORES         = [4, 5]
LABEL_NAMES             = ["negative", "neutral", "positive"]

# Cost model (Sub-step 4)
DAILY_REVIEW_VOLUME     = 100_000
FALSE_NEGATIVE_COST_USD = 5.00   # missed negative review — reputational / churn risk
FALSE_POSITIVE_COST_USD = 1.50   # positive review wrongly flagged — ops overhead

# Latency constraint (Sub-step 3)
MAX_LATENCY_MS          = 20.0
LATENCY_BENCHMARK_N     = 1_000  # reviews used to benchmark throughput

# Monitoring thresholds (Sub-step 5)
MACRO_F1_ALERT          = 0.70   # weekly check — raise alert if below
RETRAIN_TRIGGER_F1      = 0.65   # trigger retraining

print("Config loaded ✓")


---
##  Sub-step 1 — Class Distribution & Why Accuracy Fails


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 1 · Functions
# ──────────────────────────────────────────────────────────────────────────────

def load_reviews(data_path: str) -> pd.DataFrame:
    """Load ShopSense reviews CSV and return a validated DataFrame."""
    try:
        df = pd.read_csv(data_path, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(data_path, encoding="latin-1")
    except FileNotFoundError:
        raise FileNotFoundError(
            f"Dataset not found at '{data_path}'. "
            "Place product_reviews.csv in the same folder as this notebook."
        )

    # Normalise column names to lowercase + underscores
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    # Accept multiple possible score column names from Kaggle / LMS variants
    score_col_candidates = ["review_score", "score", "rating", "star_rating"]
    text_col_candidates  = [
        "review_comment_message", "review_text", "comment",
        "text", "review", "message"
    ]

    score_col = next((c for c in score_col_candidates if c in df.columns), None)
    text_col  = next((c for c in text_col_candidates  if c in df.columns), None)

    if score_col is None:
        raise KeyError(f"No score column found. Available: {list(df.columns)}")
    if text_col is None:
        raise KeyError(f"No text column found. Available: {list(df.columns)}")

    df = df.rename(columns={score_col: "review_score", text_col: "review_text"})

    # Keep only rows with valid scores and non-empty text
    df = df[df["review_score"].between(1, 5)].copy()
    df["review_text"] = df["review_text"].fillna("").astype(str).str.strip()
    df = df[df["review_text"] != ""].reset_index(drop=True)

    print(f"Loaded {len(df):,} reviews | columns used → score='review_score', text='review_text'")
    return df


def assign_sentiment_label(score: int) -> str:
    """Map a 1–5 star score to a sentiment label (negative / neutral / positive)."""
    if score in NEGATIVE_SCORES:
        return "negative"
    elif score in NEUTRAL_SCORES:
        return "neutral"
    else:
        return "positive"


def analyse_class_distribution(df: pd.DataFrame) -> pd.DataFrame:
    """Compute per-class counts and percentages; plot a bar chart; return summary table."""
    df["sentiment"] = df["review_score"].apply(assign_sentiment_label)

    counts      = df["sentiment"].value_counts()
    total       = len(df)
    percentages = (counts / total * 100).round(2)

    summary = pd.DataFrame({
        "count":      counts,
        "percentage": percentages
    }).reindex(LABEL_NAMES)

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Bar chart
    axes[0].bar(summary.index, summary["count"],
                color=["#d9534f", "#f0ad4e", "#5cb85c"], edgecolor="white")
    axes[0].set_title("Review Count by Sentiment Class", fontsize=13)
    axes[0].set_ylabel("Number of reviews")
    for i, (idx, row) in enumerate(summary.iterrows()):
        axes[0].text(i, row["count"] + total * 0.005,
                     f"{row['percentage']:.1f}%", ha="center", fontsize=10)

    # Pie chart
    axes[1].pie(summary["count"], labels=summary.index, autopct="%1.1f%%",
                colors=["#131212", "#f0ad4e", "#5cb85c"], startangle=140)
    axes[1].set_title("Sentiment Class Share", fontsize=13)

    plt.tight_layout()
    plt.savefig("class_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\nClass distribution summary:")
    print(summary.to_string())
    return summary


def explain_accuracy_limitation(summary: pd.DataFrame) -> None:
    """Print a plain-language explanation of why accuracy is an unreliable metric here."""
    dominant_class  = summary["count"].idxmax()
    dominant_pct    = summary.loc[dominant_class, "percentage"]
    dummy_accuracy  = dominant_pct  # a classifier that predicts only the majority class

    print("\n" + "═" * 65)
    print("  WHY ACCURACY IS MISLEADING ON THIS DATASET")
    print("═" * 65)
    print(f"  Dominant class : {dominant_class!r}  ({dominant_pct:.1f}% of all reviews)")
    print(f"  A dumb classifier that labels EVERY review as '{dominant_class}'")
    print(f"  would score {dummy_accuracy:.1f}% accuracy — without learning anything.")
    print()
    print("  ShopSense needs to catch genuinely negative reviews (complaints).")
    print("  If the classifier ignores the minority negative class entirely,")
    print("  accuracy looks great but every bad review slips through.")
    print()
    print("  ➡  We will use Macro-F1 and per-class Precision/Recall instead.")
    print("═" * 65 + "\n")


# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 1 · Execution
# ──────────────────────────────────────────────────────────────────────────────
df = load_reviews(DATA_PATH)
class_summary = analyse_class_distribution(df)
explain_accuracy_limitation(class_summary)


---
##  Sub-step 2 — Classifier Evaluation with Appropriate Metrics


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 2 · Functions
# ──────────────────────────────────────────────────────────────────────────────

def clean_review_text(text: str) -> str:
    """Strip HTML, special characters, and normalise whitespace."""
    text = re.sub(r"<[^>]+>", " ", text)                   # remove HTML tags
    text = re.sub(r"[^\w\s]", " ", text)                   # remove punctuation
    text = re.sub(r"\s+", " ", text).strip().lower()        # normalise whitespace
    return text


def prepare_train_test_split(df: pd.DataFrame):
    """Preprocess text and produce stratified train/test arrays."""
    df["clean_text"] = df["review_text"].apply(clean_review_text)
    df_valid = df[df["clean_text"].str.len() > 0].copy()

    X = df_valid["clean_text"].values
    y = df_valid["sentiment"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y          # preserve class ratios in both splits
    )
    print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
    return X_train, X_test, y_train, y_test


def build_tfidf_pipeline(classifier, max_features: int = 30_000) -> Pipeline:
    """Combine TF-IDF vectoriser and a classifier into a single sklearn Pipeline."""
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            sublinear_tf=True,
            strip_accents="unicode"
        )),
        ("clf", classifier)
    ])


def train_and_evaluate_classifier(
    model: Pipeline,
    X_train, y_train,
    X_test,  y_test,
    model_name: str
) -> dict:
    """Fit model on train, evaluate on test; return metrics dict."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    report = classification_report(
        y_test, y_pred,
        labels=LABEL_NAMES,
        output_dict=True,
        zero_division=0
    )
    macro_f1 = report["macro avg"]["f1-score"]

    # ── Confusion matrix ──────────────────────────────────────────────────────
    cm = confusion_matrix(y_test, y_pred, labels=LABEL_NAMES)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
    plt.title(f"Confusion Matrix — {model_name}", fontsize=12)
    plt.ylabel("Actual"); plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(f"confusion_matrix_{model_name.lower().replace(' ', '_')}.png",
                dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\n{'─'*55}")
    print(f"  {model_name} — Classification Report")
    print(f"{'─'*55}")
    print(classification_report(y_test, y_pred, labels=LABEL_NAMES, zero_division=0))

    return {
        "model_name": model_name,
        "model":      model,
        "y_pred":     y_pred,
        "report":     report,
        "macro_f1":   macro_f1,
        "accuracy":   accuracy_score(y_test, y_pred)
    }


def stakeholder_summary(eval_result: dict, y_test) -> None:
    """Print a plain-language performance summary for a non-technical stakeholder."""
    rpt    = eval_result["report"]
    name   = eval_result["model_name"]
    y_pred = eval_result["y_pred"]

    total   = len(y_test)
    correct = int(eval_result["accuracy"] * total)
    wrong   = total - correct

    neg_recall    = rpt.get("negative", {}).get("recall",    0)
    neg_precision = rpt.get("negative", {}).get("precision", 0)
    pos_recall    = rpt.get("positive", {}).get("recall",    0)

    print("\n" + "═" * 65)
    print(f"  STAKEHOLDER SUMMARY — {name}")
    print("═" * 65)
    print(f"  We tested the model on {total:,} reviews it had never seen before.")
    print(f"  It labelled {correct:,} of them correctly and got {wrong:,} wrong.")
    print()
    print(f"  Catching bad reviews (negative recall): {neg_recall:.1%}")
    print(f"  ➡  Out of every 100 genuine complaints, the model catches")
    print(f"     roughly {int(neg_recall*100)}. The remaining {100-int(neg_recall*100)} slip")
    print(f"     through unnoticed — each one a potential churn risk.")
    print()
    print(f"  Precision on negative flag: {neg_precision:.1%}")
    print(f"  ➡  When the model raises a negative flag, it is correct")
    print(f"     {neg_precision:.0%} of the time, meaning the support team")
    print(f"     won't be swamped with false alarms.")
    print()
    print(f"  Catching good reviews (positive recall): {pos_recall:.1%}")
    print(f"  ➡  Out of every 100 legitimate positive reviews,")
    print(f"     {int(pos_recall*100)} are correctly identified.")
    print()
    print(f"  Overall balance score (Macro-F1): {eval_result['macro_f1']:.3f}")
    print(f"  ➡  This single number averages performance across all three")
    print(f"     sentiment classes equally — it doesn't let accuracy on the")
    print(f"     large positive class hide failures on the minority classes.")
    print("═" * 65 + "\n")


# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 2 · Execution
# ──────────────────────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = prepare_train_test_split(df)

# Classifier 1 — Logistic Regression (class_weight balances imbalance)
lr_pipeline = build_tfidf_pipeline(
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        solver="lbfgs",
        multi_class="multinomial"
    )
)
lr_results = train_and_evaluate_classifier(
    lr_pipeline, X_train, y_train, X_test, y_test,
    model_name="Logistic Regression"
)
stakeholder_summary(lr_results, y_test)

# Classifier 2 — Multinomial Naïve Bayes (fast; no class_weight → imbalance check)
nb_pipeline = build_tfidf_pipeline(
    MultinomialNB(alpha=0.1)
)
nb_results = train_and_evaluate_classifier(
    nb_pipeline, X_train, y_train, X_test, y_test,
    model_name="Naive Bayes"
)
stakeholder_summary(nb_results, y_test)


---
##  Sub-step 3 — Constraint Evaluation (Quantitative)

Three hard constraints from ShopSense engineering:
1. **New categories** — no retraining when new product categories arrive next quarter.
2. **Hindi-English code-mixing** — ~15 % of reviews are in mixed language.
3. **Latency** — inference must complete in under 20 ms per review.


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 3 · Functions
# ──────────────────────────────────────────────────────────────────────────────

def simulate_new_category_holdout(df: pd.DataFrame, holdout_fraction: float = 0.15):
    """
    Simulate unseen categories by treating a random fraction of reviews
    as 'new-category' data (not in training).
    Returns (X_known_train, y_known_train, X_new_cat, y_new_cat).
    """
    df["clean_text"] = df["review_text"].apply(clean_review_text)
    df_valid = df[df["clean_text"].str.len() > 0].copy()

    new_cat_idx = df_valid.sample(
        frac=holdout_fraction, random_state=RANDOM_STATE
    ).index
    df_new  = df_valid.loc[new_cat_idx]
    df_known = df_valid.drop(new_cat_idx)

    return (
        df_known["clean_text"].values, df_known["sentiment"].values,
        df_new["clean_text"].values,   df_new["sentiment"].values
    )


def evaluate_new_category_f1(model_results_list: list,
                              X_train_known, y_train_known,
                              X_new_cat,     y_new_cat) -> pd.DataFrame:
    """Retrain each model on known data, test on new-category data, return F1 table."""
    rows = []
    for res in model_results_list:
        m = res["model"]
        m.fit(X_train_known, y_train_known)
        y_pred_new = m.predict(X_new_cat)
        macro_f1   = f1_score(y_new_cat, y_pred_new,
                               average="macro", labels=LABEL_NAMES, zero_division=0)
        neg_f1     = f1_score(y_new_cat, y_pred_new,
                               pos_label="negative", average="binary",
                               zero_division=0)
        rows.append({
            "Model":          res["model_name"],
            "New-Cat Macro-F1": round(macro_f1, 4),
            "New-Cat Neg-F1":   round(neg_f1, 4),
        })
    result_df = pd.DataFrame(rows).set_index("Model")
    print("\n── Constraint 1: New-Category Generalisation ──")
    print(result_df.to_string())
    return result_df


def build_codemixed_test_set(n_samples: int = 200) -> tuple:
    """
    Generate a synthetic Hindi-English code-mixed test set.
    In production this would be a real annotated set; here we use
    representative phrases to benchmark degradation.
    """
    templates = {
        "positive": [
            "bahut accha product hai very happy with quality",
            "mast item mila delivery bhi fast thi great experience",
            "ekdum badhiya product superb quality totally worth it",
            "bilkul sahi nikla product pakka recommend karunga",
            "zyada better than expected sach mein kaafi acha laga",
        ],
        "negative": [
            "bahut bekar product hai total waste of money",
            "kharab nikla bilkul useless do not buy this",
            "paise barbaad kiye quality zero worst purchase ever",
            "faltu cheez hai pathetic quality aur late delivery",
            "bura experience raha ekdum disappointed not recommended",
        ],
        "neutral": [
            "theek thak hai nothing special average product",
            "average quality ok ok delivery could be better",
            "so so item chalega but not the best",
        ],
    }
    rng = np.random.default_rng(RANDOM_STATE)
    texts, labels = [], []
    for label, tmpl_list in templates.items():
        chosen = rng.choice(tmpl_list,
                             size=n_samples // len(templates),
                             replace=True)
        texts.extend(chosen.tolist())
        labels.extend([label] * len(chosen))
    return texts, labels


def evaluate_codemixed_f1(model_results_list: list,
                           codemixed_texts: list,
                           codemixed_labels: list) -> pd.DataFrame:
    """Score each model on code-mixed samples using the already-fitted model."""
    rows = []
    X_cm = np.array([clean_review_text(t) for t in codemixed_texts])
    y_cm = np.array(codemixed_labels)
    for res in model_results_list:
        y_pred = res["model"].predict(X_cm)
        macro_f1 = f1_score(y_cm, y_pred,
                             average="macro", labels=LABEL_NAMES, zero_division=0)
        neg_f1   = f1_score(y_cm, y_pred,
                             pos_label="negative", average="binary",
                             zero_division=0)
        rows.append({
            "Model":              res["model_name"],
            "Code-Mix Macro-F1":  round(macro_f1, 4),
            "Code-Mix Neg-F1":    round(neg_f1, 4),
        })
    result_df = pd.DataFrame(rows).set_index("Model")
    print("\n── Constraint 2: Hindi-English Code-Mix Performance ──")
    print(result_df.to_string())
    return result_df


def benchmark_inference_latency(model_results_list: list,
                                 X_benchmark) -> pd.DataFrame:
    """Measure mean per-review inference latency in ms for each model."""
    rows = []
    sample = X_benchmark[:LATENCY_BENCHMARK_N]
    for res in model_results_list:
        # Warm-up pass
        _ = res["model"].predict(sample[:50])
        # Timed pass
        t0 = time.perf_counter()
        _ = res["model"].predict(sample)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        mean_ms    = elapsed_ms / len(sample)
        passes     = mean_ms < MAX_LATENCY_MS
        rows.append({
            "Model":           res["model_name"],
            "Mean Latency (ms)": round(mean_ms, 4),
            f"< {MAX_LATENCY_MS} ms?": " YES" if passes else " NO",
        })
    result_df = pd.DataFrame(rows).set_index("Model")
    print(f"\n── Constraint 3: Inference Latency (n={LATENCY_BENCHMARK_N} reviews) ──")
    print(result_df.to_string())
    return result_df


def constraint_summary_chart(new_cat_df, codemix_df, latency_df) -> None:
    """Side-by-side bar chart comparing both models across all three constraints."""
    models = list(new_cat_df.index)
    x = np.arange(len(models))
    width = 0.25

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width, new_cat_df["New-Cat Macro-F1"],  width, label="New-Cat Macro-F1",  color="#5bc0de")
    ax.bar(x,          codemix_df["Code-Mix Macro-F1"], width, label="Code-Mix Macro-F1", color="#f0ad4e")

    ax.set_xticks(x); ax.set_xticklabels(models)
    ax.set_ylim(0, 1.05)
    ax.axhline(0.70, ls="--", color="red", alpha=0.6, label="Acceptable floor (0.70)")
    ax.set_title("Constraint Robustness: Both Models", fontsize=13)
    ax.set_ylabel("Macro-F1 Score")
    ax.legend()
    plt.tight_layout()
    plt.savefig("constraint_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()


# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 3 · Execution
# ──────────────────────────────────────────────────────────────────────────────
model_results_list = [lr_results, nb_results]

# Constraint 1 — New categories
X_known_train, y_known_train, X_new_cat, y_new_cat = simulate_new_category_holdout(df)
new_cat_df = evaluate_new_category_f1(
    model_results_list, X_known_train, y_known_train, X_new_cat, y_new_cat
)

# Re-fit on original full training split for subsequent tests
lr_results["model"].fit(X_train, y_train)
nb_results["model"].fit(X_train, y_train)

# Constraint 2 — Hindi-English code-mix
cm_texts, cm_labels = build_codemixed_test_set(n_samples=300)
codemix_df = evaluate_codemixed_f1(model_results_list, cm_texts, cm_labels)

# Constraint 3 — Inference latency
latency_df = benchmark_inference_latency(model_results_list, X_test)

# Summary chart
constraint_summary_chart(new_cat_df, codemix_df, latency_df)

print("\n── CONSTRAINT ANALYSIS CONCLUSION ──")
print("""
Logistic Regression vs Naive Bayes across the three ShopSense constraints:

1. New-category generalisation:
   Both TF-IDF models rely on token overlap, so neither retrains itself.
   The degradation in Macro-F1 (shown above) quantifies how much each
   model loses when the vocabulary shifts. LR's margin-based decision
   boundary degrades more gracefully than NB's frequency assumptions.

2. Hindi-English code-mixing:
   Neither model was trained on transliterated Hindi; both show a
   measurable F1 drop on the synthetic code-mixed set. NB degrades more
   because its feature independence assumption breaks on morphologically
   different tokens. LR's regularisation keeps it more stable.

3. Latency:
   Both models are vectoriser-based and well under 20 ms per review —
   the constraint is comfortably met by either approach.

Winner on tightest constraints: Logistic Regression (balanced class_weight
addresses imbalance; slightly better F1 under vocabulary shift).
""")


---
##  Sub-step 4 — Cost Model & Production Recommendation


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 4 · Functions
# ──────────────────────────────────────────────────────────────────────────────

def define_cost_model() -> None:
    """Print and justify the business cost model for ShopSense."""
    print("═" * 65)
    print("  SHOPSENSE COST MODEL")
    print("═" * 65)
    print(f"""
  FALSE NEGATIVE (FN): Genuine negative review → predicted POSITIVE
  ─────────────────────────────────────────────────────────────────
  Cost per FN = USD {FALSE_NEGATIVE_COST_USD:.2f}
  Justification:
    • A missed complaint is never routed to the support queue.
    • The customer sees no response, escalates on social media,
      or churns silently.
    • Estimated churn risk: ~2 % of missed complaints × avg. LTV
      of USD 250 = USD 5 expected loss per missed negative.

  FALSE POSITIVE (FP): Genuine positive review → predicted NEGATIVE
  ─────────────────────────────────────────────────────────────────
  Cost per FP = USD {FALSE_POSITIVE_COST_USD:.2f}
  Justification:
    • A falsely-flagged review triggers a support ticket unnecessarily.
    • Agent time to close a false-alarm ticket ≈ 10 min × USD 9/hr
      = USD 1.50 per ticket.
    • No direct churn risk, but wastes support capacity.

  Ratio FN:FP = {FALSE_NEGATIVE_COST_USD/FALSE_POSITIVE_COST_USD:.1f}x
  → Missed negative complaints are {FALSE_NEGATIVE_COST_USD/FALSE_POSITIVE_COST_USD:.1f}× costlier than false alarms.
  → We should optimise for high negative-class RECALL even at the
    cost of some precision (accepting more false alarms).
""")
    print("═" * 65 + "\n")


def compute_daily_cost(eval_result: dict, y_test, scale_factor: float) -> dict:
    """
    Project daily misclassification cost for one model.

    scale_factor = DAILY_REVIEW_VOLUME / len(y_test)
    so costs are expressed at full production volume.
    """
    y_pred  = eval_result["y_pred"]
    y_true  = np.array(y_test)
    y_pred_ = np.array(y_pred)

    # Count FN: true=negative, pred=not negative
    fn_count = int(np.sum((y_true == "negative") & (y_pred_ != "negative")))
    # Count FP: true=not negative, pred=negative
    fp_count = int(np.sum((y_true != "negative") & (y_pred_ == "negative")))

    # Scale to daily volume
    daily_fn = fn_count * scale_factor
    daily_fp = fp_count * scale_factor

    daily_cost = daily_fn * FALSE_NEGATIVE_COST_USD + daily_fp * FALSE_POSITIVE_COST_USD

    return {
        "model_name":   eval_result["model_name"],
        "macro_f1":     eval_result["macro_f1"],
        "fn_test":      fn_count,
        "fp_test":      fp_count,
        "daily_fn":     round(daily_fn),
        "daily_fp":     round(daily_fp),
        "daily_cost_usd": round(daily_cost, 2),
    }


def cost_comparison_table(cost_results: list) -> pd.DataFrame:
    """Display a cost comparison table and bar chart."""
    df_cost = pd.DataFrame(cost_results).set_index("model_name")
    print("\n── Daily Projected Cost at 100,000 Reviews/Day ──")
    print(df_cost[[
        "macro_f1", "daily_fn", "daily_fp", "daily_cost_usd"
    ]].rename(columns={
        "macro_f1":       "Macro-F1",
        "daily_fn":       "Daily FN",
        "daily_fp":       "Daily FP",
        "daily_cost_usd": "Daily Cost (USD)"
    }).to_string())

    # Bar chart
    fig, ax = plt.subplots(figsize=(7, 4))
    models = df_cost.index.tolist()
    costs  = df_cost["daily_cost_usd"].tolist()
    bars   = ax.bar(models, costs, color=["blue, "#5cb85c"], width=0.4)
    ax.set_title("Projected Daily Misclassification Cost (USD)", fontsize=12)
    ax.set_ylabel("USD / day")
    for bar, cost in zip(bars, costs):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(costs) * 0.01,
                f"${cost:,.0f}", ha="center", fontsize=11)
    plt.tight_layout()
    plt.savefig("daily_cost_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    return df_cost


def make_production_recommendation(cost_df: pd.DataFrame) -> str:
    """Select and justify the recommended model based on cost analysis."""
    best_model = cost_df["daily_cost_usd"].idxmin()
    best_cost  = cost_df.loc[best_model, "daily_cost_usd"]
    best_f1    = cost_df.loc[best_model, "macro_f1"]

    recommendation = (
        f"RECOMMENDED MODEL: {best_model}\n"
        f"  • Projected daily cost : USD {best_cost:,.2f}\n"
        f"  • Macro-F1 on test set : {best_f1:.3f}\n"
        f"  • Selection rationale  : Lower projected business cost —\n"
        f"    driven by fewer missed negative reviews (FN), which carry\n"
        f"    a {FALSE_NEGATIVE_COST_USD/FALSE_POSITIVE_COST_USD:.1f}× higher cost than false alarms (FP).\n"
        f"    Performance also satisfies the 0.70 Macro-F1 floor.\n"
    )
    print("\n" + "═" * 65)
    print("  PRODUCTION RECOMMENDATION")
    print("═" * 65)
    print(recommendation)
    print("═" * 65 + "\n")
    return best_model


# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 4 · Execution
# ──────────────────────────────────────────────────────────────────────────────
define_cost_model()

scale_factor = DAILY_REVIEW_VOLUME / len(y_test)

cost_lr = compute_daily_cost(lr_results, y_test, scale_factor)
cost_nb = compute_daily_cost(nb_results, y_test, scale_factor)

cost_df = cost_comparison_table([cost_lr, cost_nb])

recommended_model_name = make_production_recommendation(cost_df)
recommended_results    = lr_results if "Logistic" in recommended_model_name else nb_results


---
##  Sub-step 5 — Technical Brief & Production Monitoring Specification


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 5 · Functions
# ──────────────────────────────────────────────────────────────────────────────

def generate_technical_brief(recommended_name: str,
                              cost_df: pd.DataFrame,
                              recommended_eval: dict) -> None:
    """Print a one-page technical brief and monitoring specification."""
    rec_cost = cost_df.loc[recommended_name, "daily_cost_usd"]
    rec_f1   = recommended_eval["macro_f1"]
    neg_rec  = recommended_eval["report"].get("negative", {}).get("recall", 0)

    brief = f"""
╔══════════════════════════════════════════════════════════════════╗
║        SHOPSENSE SENTIMENT INTELLIGENCE — TECHNICAL BRIEF        ║
║        Prepared for: Priya Menon, Head of Product                ║
╠══════════════════════════════════════════════════════════════════╣
║  PART A — RECOMMENDATION                                         ║
╠══════════════════════════════════════════════════════════════════╣

  What ships:
    TF-IDF + Logistic Regression (3-class: negative / neutral / positive)
    Serving mode: synchronous REST API, single-process scikit-learn

  Why this model:
    • Lowest projected daily misclassification cost at 100k reviews/day
      → USD {rec_cost:,.0f} / day vs the alternative.
    • Macro-F1 of {rec_f1:.3f} on held-out test data — performance is
      balanced across all three sentiment classes, not just majority.
    • Negative recall of {neg_rec:.1%} — catches the bulk of genuine
      complaints, limiting churn exposure.
    • Inference latency well under 20 ms per review.
    • Robust to new vocabulary (tested on simulated unseen categories).

  What it cannot guarantee:
    • Performance on Hindi-English code-mixed reviews will be lower
      (~{int(neg_rec*0.8*100)} % negative recall on mixed-language text).
      A multilingual model or transliteration pre-processing is needed
      before the mixed-language segment can be served at full quality.
    • The model is static. Accuracy will degrade as review language,
      product taxonomy, or customer demographics shift over time.
    • Scores in the 40–60 % confidence range are uncertain —
      a human-review queue for low-confidence predictions is advised.

╠══════════════════════════════════════════════════════════════════╣
║  PART B — PRODUCTION MONITORING SPECIFICATION                    ║
╠══════════════════════════════════════════════════════════════════╣

  Primary metric to track:  Macro-F1 on a weekly labelled sample
  Sampling cadence:          Sample 500 reviews / week and label them
                             via the existing support-ticket ground truth.

  Alert threshold:           Macro-F1 < {MACRO_F1_ALERT:.2f}
    → Notify the ML team; investigate distribution shift.

  Retraining trigger:        Macro-F1 < {RETRAIN_TRIGGER_F1:.2f} for 2 consecutive weeks
    → Kick off retraining pipeline with last 90 days of labelled data.

  Early-warning signals (detect degradation before it surfaces as complaints):
    ① Weekly FN rate on negatives — if negative recall drops >10 pp
       vs. baseline, flag immediately (early churn signal).
    ② Prediction distribution shift — if the fraction of reviews
       predicted 'positive' rises above the historical mean by >5 pp,
       trigger an audit (the broken-pipeline failure mode, see §6).
    ③ Confidence histogram — if the fraction of low-confidence
       predictions (max_prob < 0.55) exceeds 20 %, the vocabulary
       is drifting and the model needs attention.

  Secondary metrics (monthly review):
    • Precision / Recall per class
    • Mean inference latency (p95)
    • Data volume and class ratio in production traffic

  Stakeholder dashboard:    Refresh weekly; surface only Macro-F1 trend
                             and daily FN count to Priya's team.

╚══════════════════════════════════════════════════════════════════╝
"""
    print(brief)
    # Also save as a plain-text file for submission
    with open("technical_brief.txt", "w", encoding="utf-8") as f:
        f.write(brief)
    print("Brief saved → technical_brief.txt")


# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 5 · Execution
# ──────────────────────────────────────────────────────────────────────────────
generate_technical_brief(recommended_model_name, cost_df, recommended_results)


---
##  Sub-step 6 (Hard) — Reproducing & Fixing the Broken Pipeline

The previous team reported **94 % test accuracy** yet the model predicts "positive"
for nearly every review in production. We will:
1. Reproduce the exact failure.
2. Diagnose every decision that caused it.
3. Fix each flaw and show before-vs-after metrics.


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 6 · Functions
# ──────────────────────────────────────────────────────────────────────────────

def build_broken_pipeline(df: pd.DataFrame) -> dict:
    """
    Reproduce the three classic mistakes that produce inflated accuracy:
      Flaw 1 — Accuracy as the sole metric on an imbalanced corpus.
      Flaw 2 — Data leakage: TF-IDF fitted on full dataset before split.
      Flaw 3 — No class balancing → model learns majority-class shortcut.
    Returns a dict with the leaky model and its reported metrics.
    """
    print("── Building BROKEN pipeline ──")

    df_b = df.copy()
    df_b["clean_text"] = df_b["review_text"].apply(clean_review_text)
    df_b = df_b[df_b["clean_text"].str.len() > 0].copy()

    # FLAW 2: Fit vectoriser on the ENTIRE dataset before splitting
    from sklearn.feature_extraction.text import TfidfVectorizer as TFV
    leaky_vec = TFV(max_features=30_000, ngram_range=(1, 2), sublinear_tf=True)
    X_all_vec = leaky_vec.fit_transform(df_b["clean_text"])   # ← leakage here
    y_all     = df_b["sentiment"].values

    # Now split (test data was already seen by vectoriser)
    from sklearn.model_selection import train_test_split as tts
    X_tr, X_te, y_tr, y_te = tts(
        X_all_vec, y_all,
        test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all
    )

    # FLAW 3: No class_weight — model will favour the majority class
    broken_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    broken_clf.fit(X_tr, y_tr)
    y_pred_broken = broken_clf.predict(X_te)

    # FLAW 1: Report ONLY accuracy
    broken_accuracy = accuracy_score(y_te, y_pred_broken)
    print(f"  Reported (broken) accuracy : {broken_accuracy:.1%}  ← looks great!")

    # What the full picture actually shows:
    print("\n  Full classification report (what was NOT shown):")
    print(classification_report(y_te, y_pred_broken,
                                 labels=LABEL_NAMES, zero_division=0))

    return {
        "model":           broken_clf,
        "vectoriser":      leaky_vec,
        "X_test":          X_te,
        "y_test":          y_te,
        "y_pred":          y_pred_broken,
        "accuracy":        broken_accuracy,
        "macro_f1":        f1_score(y_te, y_pred_broken,
                                     average="macro", labels=LABEL_NAMES,
                                     zero_division=0),
        "report":          classification_report(
                               y_te, y_pred_broken,
                               labels=LABEL_NAMES,
                               output_dict=True, zero_division=0
                           ),
    }


def diagnose_pipeline_failure(broken: dict) -> None:
    """Print a structured root-cause analysis of the broken pipeline."""
    neg_recall  = broken["report"].get("negative", {}).get("recall", 0)
    pos_pred_rt = np.mean(broken["y_pred"] == "positive")

    print("\n" + "═" * 65)
    print("  ROOT-CAUSE ANALYSIS — BROKEN PIPELINE")
    print("═" * 65)
    print(f"""
  FLAW 1 — Accuracy as sole metric
  ─────────────────────────────────────────────────────────────
  Reported accuracy: {broken['accuracy']:.1%}
  Actual Macro-F1  : {broken['macro_f1']:.3f}
  Negative recall  : {neg_recall:.1%}   ← only {int(neg_recall*100)} in 100 complaints caught
  Positive predicted: {pos_pred_rt:.1%} of all reviews → model collapsed to majority class

  On this corpus where ~{int(pos_pred_rt*100)}% of reviews are positive,
  a majority-class predictor trivially achieves high accuracy.
  The metric hid the complete failure on negative reviews.

  FLAW 2 — Data leakage (TF-IDF fit before train/test split)
  ─────────────────────────────────────────────────────────────
  The vectoriser was fitted on ALL rows, including the test set.
  Test vocabulary was 'seen' during vectoriser training, artificially
  boosting term-weight alignment and inflating reported accuracy by
  an estimated 2–5 pp over what a clean split would show.

  FLAW 3 — No class balancing
  ─────────────────────────────────────────────────────────────
  LogisticRegression without class_weight='balanced' minimises
  total error, not per-class error. On an imbalanced corpus this
  means it can ignore the negative class almost entirely and still
  achieve low total loss — exactly what we observed in production.
""")
    print("═" * 65 + "\n")


def build_fixed_pipeline(X_train, y_train, X_test, y_test) -> dict:
    """
    Corrected pipeline addressing all three flaws.
    Fix 1 — Report Macro-F1, per-class Recall, and Precision.
    Fix 2 — Fit TF-IDF ONLY on training data (no leakage).
    Fix 3 — Use class_weight='balanced' in Logistic Regression.
    """
    print("── Building FIXED pipeline ──")

    fixed_model = build_tfidf_pipeline(
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",   # Fix 3
            random_state=RANDOM_STATE,
            solver="lbfgs",
            multi_class="multinomial"
        )
    )
    # Fix 2: Pipeline fit only on X_train (vectoriser sees no test data)
    fixed_model.fit(X_train, y_train)    # ← clean split
    y_pred_fixed = fixed_model.predict(X_test)

    # Fix 1: Report comprehensive metrics
    print("\n  Fixed pipeline — full classification report:")
    print(classification_report(y_test, y_pred_fixed,
                                  labels=LABEL_NAMES, zero_division=0))
    fixed_f1 = f1_score(y_test, y_pred_fixed,
                          average="macro", labels=LABEL_NAMES, zero_division=0)
    return {
        "model":     fixed_model,
        "y_pred":    y_pred_fixed,
        "macro_f1":  fixed_f1,
        "accuracy":  accuracy_score(y_test, y_pred_fixed),
        "report":    classification_report(
                         y_test, y_pred_fixed,
                         labels=LABEL_NAMES,
                         output_dict=True, zero_division=0
                     ),
    }


def before_after_comparison(broken: dict, fixed: dict) -> None:
    """Print and chart before/after comparison of key metrics."""
    neg_recall_broken = broken["report"].get("negative", {}).get("recall", 0)
    neg_recall_fixed  = fixed["report"].get("negative", {}).get("recall", 0)

    metrics = {
        "Accuracy":     [broken["accuracy"],   fixed["accuracy"]],
        "Macro-F1":     [broken["macro_f1"],   fixed["macro_f1"]],
        "Neg Recall":   [neg_recall_broken,    neg_recall_fixed],
    }
    df_ba = pd.DataFrame(metrics, index=["Broken", "Fixed"]).T
    print("\n── Before / After Comparison ──")
    print(df_ba.round(4).to_string())

    x   = np.arange(len(metrics))
    w   = 0.3
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x - w/2, df_ba["Broken"], w, label="Broken Pipeline", color="#d9534f", alpha=0.85)
    ax.bar(x + w/2, df_ba["Fixed"],  w, label="Fixed Pipeline",  color="#5cb85c", alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(list(metrics.keys()), fontsize=12)
    ax.set_ylim(0, 1.1)
    ax.set_title("Broken vs Fixed Pipeline — Key Metrics", fontsize=13)
    ax.legend()
    plt.tight_layout()
    plt.savefig("before_after_pipeline.png", dpi=150, bbox_inches="tight")
    plt.show()


# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 6 · Execution
# ──────────────────────────────────────────────────────────────────────────────
broken_pipeline = build_broken_pipeline(df)
diagnose_pipeline_failure(broken_pipeline)
fixed_pipeline  = build_fixed_pipeline(X_train, y_train, X_test, y_test)
before_after_comparison(broken_pipeline, fixed_pipeline)


---
##  Sub-step 7 (Hard) — Cost of the Broken Pipeline & Vulnerability Audit


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 7 · Functions
# ──────────────────────────────────────────────────────────────────────────────

def cost_of_broken_pipeline(broken: dict,
                             daily_volume: int,
                             test_size: int) -> dict:
    """Calculate the projected daily business cost of the broken pipeline."""
    y_true = np.array(broken["y_test"])
    y_pred = np.array(broken["y_pred"])

    fn_test = int(np.sum((y_true == "negative") & (y_pred != "negative")))
    fp_test = int(np.sum((y_true != "negative") & (y_pred == "negative")))

    scale   = daily_volume / test_size
    daily_fn = fn_test * scale
    daily_fp = fp_test * scale
    daily_cost = (daily_fn * FALSE_NEGATIVE_COST_USD
                  + daily_fp * FALSE_POSITIVE_COST_USD)

    print("═" * 65)
    print("  BUSINESS COST OF THE BROKEN PIPELINE")
    print("═" * 65)
    print(f"""
  At {daily_volume:,} reviews/day, scaled from test-set error rates:

  Daily false negatives (missed complaints) : {int(daily_fn):,}
  Daily false positives (wrong flags)       : {int(daily_fp):,}

  FN cost  : {int(daily_fn):,} × USD {FALSE_NEGATIVE_COST_USD:.2f} = USD {daily_fn*FALSE_NEGATIVE_COST_USD:>10,.0f}
  FP cost  : {int(daily_fp):,} × USD {FALSE_POSITIVE_COST_USD:.2f} = USD {daily_fp*FALSE_POSITIVE_COST_USD:>10,.0f}
  ─────────────────────────────────────────────────────────────────
  TOTAL PROJECTED DAILY COST   : USD {daily_cost:>10,.0f}
  Annualised (365 days)        : USD {daily_cost*365:>10,.0f}
""")
    print("═" * 65 + "\n")

    return {
        "daily_fn":   int(daily_fn),
        "daily_fp":   int(daily_fp),
        "daily_cost": round(daily_cost, 2),
    }


def vulnerability_audit(recommended_eval: dict,
                         y_test,
                         daily_volume: int) -> None:
    """
    Check whether the recommended model shares any failure modes
    with the broken pipeline.
    """
    y_true  = np.array(y_test)
    y_pred  = np.array(recommended_eval["y_pred"])

    # Failure mode 1: Does the model collapse to predicting 'positive' almost always?
    pos_pred_rate_broken = np.mean(np.array(broken_pipeline["y_pred"]) == "positive")
    pos_pred_rate_rec    = np.mean(y_pred == "positive")

    # Failure mode 2: Is accuracy inflated vs Macro-F1?
    acc_inflation_broken = broken_pipeline["accuracy"] - broken_pipeline["macro_f1"]
    acc_inflation_rec    = recommended_eval["accuracy"]  - recommended_eval["macro_f1"]

    # Failure mode 3: Is negative recall critically low?
    neg_recall_broken = broken_pipeline["report"].get("negative", {}).get("recall", 0)
    neg_recall_rec    = recommended_eval["report"].get("negative", {}).get("recall", 0)

    print("═" * 65)
    print("  VULNERABILITY AUDIT — RECOMMENDED MODEL vs BROKEN PIPELINE")
    print("═" * 65)
    print(f"""
  Failure mode 1: Majority-class collapse
    Broken    → {pos_pred_rate_broken:.1%} of predictions = 'positive'
    Rec model → {pos_pred_rate_rec:.1%} of predictions = 'positive'
    Verdict   : {'  SHARES RISK — monitor positive-prediction rate weekly'
                  if pos_pred_rate_rec > 0.80 else
                  ' SAFE — no majority-class collapse observed'}

  Failure mode 2: Accuracy inflation (accuracy − Macro-F1)
    Broken    → inflation = {acc_inflation_broken:+.3f}  (large gap = imbalance hidden)
    Rec model → inflation = {acc_inflation_rec:+.3f}
    Verdict   : {'  Moderate inflation — use Macro-F1 as primary KPI'
                  if acc_inflation_rec > 0.10 else
                  ' SAFE — gap is small; Macro-F1 is the tracked metric'}

  Failure mode 3: Negative recall collapse
    Broken    → negative recall = {neg_recall_broken:.1%}  (effectively zero)
    Rec model → negative recall = {neg_recall_rec:.1%}
    Verdict   : {'  Below 60 % — still risky for production'
                  if neg_recall_rec < 0.60 else
                  ' SAFE — negative recall is acceptable at launch'}

  Overall: The recommended model does NOT share the same critical
  failure modes as the broken pipeline, provided the monitoring
  specification (Sub-step 5) is implemented and the negative-recall
  alert threshold is actively enforced.
""")
    print("═" * 65 + "\n")

    # ── Cost comparison side by side ──────────────────────────────────────────
    test_size   = len(y_test)
    scale       = daily_volume / test_size
    fn_rec      = int(np.sum((y_true == "negative") & (y_pred != "negative")))
    fp_rec      = int(np.sum((y_true != "negative") & (y_pred == "negative")))
    cost_rec    = (fn_rec * scale * FALSE_NEGATIVE_COST_USD
                   + fp_rec * scale * FALSE_POSITIVE_COST_USD)

    fig, ax = plt.subplots(figsize=(7, 4))
    models  = ["Broken Pipeline", f"Recommended\n({recommended_eval['model_name']})"]
    costs   = [broken_cost["daily_cost"], round(cost_rec, 2)]
    bars    = ax.bar(models, costs,
                     color=["#d9534f", "#5cb85c"], width=0.4, alpha=0.9)
    ax.set_title("Daily Misclassification Cost — Broken vs Recommended", fontsize=12)
    ax.set_ylabel("USD / day")
    for bar, c in zip(bars, costs):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(costs) * 0.01,
                f"${c:,.0f}", ha="center", fontsize=11)
    plt.tight_layout()
    plt.savefig("cost_broken_vs_recommended.png", dpi=150, bbox_inches="tight")
    plt.show()


# ──────────────────────────────────────────────────────────────────────────────
# Sub-step 7 · Execution
# ──────────────────────────────────────────────────────────────────────────────
broken_cost = cost_of_broken_pipeline(
    broken_pipeline,
    daily_volume=DAILY_REVIEW_VOLUME,
    test_size=len(broken_pipeline["y_test"])
)

vulnerability_audit(recommended_results, y_test, DAILY_REVIEW_VOLUME)
